# Experiment 1: HAM10000 Baseline Training

Train MedSigLIP on HAM10000 dermoscopic images with:
- Frozen vision encoder (only 596K trainable params)
- Focal Loss with 2x melanoma weight
- Hybrid early stopping (melanoma recall + balanced accuracy + F1)

**Results:** 78.9% balanced accuracy, 85.7% melanoma recall on HAM10000 val.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kabirgrover/dermtriage/blob/master/notebooks/01_train_baseline.ipynb)

In [ ]:
# Install dependencies
!pip install -q torch torchvision transformers scikit-learn tqdm kaggle huggingface-hub

In [ ]:
# Authenticate
import os
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
# Download HAM10000
import subprocess
import shutil
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_DIR = Path('/content/data')
PROCESSED_DIR = DATA_DIR / 'processed'
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

if not (PROCESSED_DIR / 'train').exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'kmader/skin-cancer-mnist-ham10000',
                    '-p', str(DATA_DIR), '--unzip'], check=True)

    df = pd.read_csv(DATA_DIR / 'HAM10000_metadata.csv')
    img_dirs = [DATA_DIR / 'HAM10000_images_part_1', DATA_DIR / 'HAM10000_images_part_2']
    for split in ['train', 'val']:
        for cls in CLASS_NAMES:
            (PROCESSED_DIR / split / cls).mkdir(parents=True, exist_ok=True)

    train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['dx'], random_state=42)
    for split_df, split_name in [(train_df, 'train'), (val_df, 'val')]:
        for _, row in split_df.iterrows():
            img_name = f"{row['image_id']}.jpg"
            for img_dir in img_dirs:
                src = img_dir / img_name
                if src.exists():
                    shutil.copy(src, PROCESSED_DIR / split_name / row['dx'] / img_name)
                    break
    print('Dataset ready!')
else:
    print('Dataset already prepared!')

In [ ]:
# Imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from collections import Counter
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score, balanced_accuracy_score

NUM_CLASSES = len(CLASS_NAMES)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Model
class MedSigLIPClassifier(nn.Module):
    def __init__(self, num_classes=7, dropout_rate=0.3, freeze_encoder=True):
        super().__init__()
        from transformers import AutoModel
        full_model = AutoModel.from_pretrained('google/medsiglip-448', torch_dtype=torch.float32)
        self.vision_model = full_model.vision_model
        self.embed_dim = full_model.config.vision_config.hidden_size
        del full_model
        if freeze_encoder:
            for param in self.vision_model.parameters():
                param.requires_grad = False
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.embed_dim), nn.Dropout(dropout_rate),
            nn.Linear(self.embed_dim, 512), nn.GELU(), nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values):
        with torch.no_grad():
            outputs = self.vision_model(pixel_values=pixel_values)
            features = outputs.pooler_output if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None else outputs.last_hidden_state.mean(dim=1)
        return self.classifier(features)


class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, class_weights=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.class_weights = class_weights

    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(weight=self.class_weights, reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)
        return (self.alpha * (1 - pt) ** self.gamma * ce_loss).mean()


class HAM10000Dataset(Dataset):
    def __init__(self, data_dir, split='train', transform=None):
        self.transform = transform
        self.class_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
        self.samples = []
        split_dir = Path(data_dir) / split
        for class_name in CLASS_NAMES:
            class_dir = split_dir / class_name
            if class_dir.exists():
                for img_path in class_dir.glob('*.jpg'):
                    self.samples.append((img_path, self.class_to_idx[class_name]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, label

print('Classes defined!')

In [ ]:
# Transforms & DataLoaders
train_transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
val_transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = HAM10000Dataset(str(PROCESSED_DIR), split='train', transform=train_transform)
val_dataset = HAM10000Dataset(str(PROCESSED_DIR), split='val', transform=val_transform)
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

# Class weights with 2x melanoma boost
train_labels = [label for _, label in train_dataset]
label_counts = Counter(train_labels)
total = len(train_labels)
class_weights_dict = {}
for i in range(NUM_CLASSES):
    count = label_counts.get(i, 1)
    base = np.sqrt(total / (NUM_CLASSES * count))
    class_weights_dict[i] = base * 2.0 if CLASS_NAMES[i] == 'mel' else base
class_weights = torch.tensor([class_weights_dict[i] for i in range(NUM_CLASSES)], dtype=torch.float32, device=device)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
# Training
model = MedSigLIPClassifier(num_classes=NUM_CLASSES, dropout_rate=0.3, freeze_encoder=True).to(device)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)
criterion = FocalLoss(alpha=1.0, gamma=2.0, class_weights=class_weights)

best_hybrid_score = 0.0
patience, max_patience = 0, 10
CHECKPOINT_DIR = Path('/content/checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

for epoch in range(30):
    model.train()
    running_loss = 0.0
    train_preds, train_labels_list = [], []
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/30')

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        if torch.isnan(loss) or torch.isinf(loss): continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
        train_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    # Validation
    model.eval()
    val_preds, val_labels_list = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            val_labels_list.extend(labels.cpu().numpy())

    val_bal_acc = balanced_accuracy_score(val_labels_list, val_preds)
    val_f1 = f1_score(val_labels_list, val_preds, average='macro')
    mel_idx = CLASS_NAMES.index('mel')
    mel_recall = recall_score(val_labels_list, val_preds, labels=[mel_idx], average='micro', zero_division=0)
    hybrid_score = 0.5 * mel_recall + 0.3 * val_bal_acc + 0.2 * val_f1

    print(f'Epoch {epoch+1}: Bal Acc={val_bal_acc:.3f}, Mel Recall={mel_recall:.3f}, Hybrid={hybrid_score:.4f}')
    scheduler.step()

    if hybrid_score > best_hybrid_score:
        best_hybrid_score = hybrid_score
        patience = 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'classifier_state_dict': model.classifier.state_dict(),
                    'hybrid_score': hybrid_score, 'mel_recall': mel_recall,
                    'val_balanced_acc': val_bal_acc}, CHECKPOINT_DIR / 'best_model.pth')
        print(f'  *** New best: {hybrid_score:.4f} ***')
    else:
        patience += 1
        if patience >= max_patience:
            print('Early stopping!')
            break

print(f'\nDone! Best hybrid score: {best_hybrid_score:.4f}')

## Results (Experiment 1)

| Metric | HAM10000 Val |
|--------|-------------|
| Balanced Accuracy | **78.9%** |
| Melanoma Recall | **85.7%** |
| Macro F1 | 72.2% |

The model achieves strong in-domain performance, but evaluation on PAD-UFES-20 clinical images reveals a significant domain gap (see Experiment 2).